In [1]:
from sklearn.metrics import log_loss, roc_auc_score
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from __future__ import annotations
from pathlib import Path
from typing import Union
import xgboost as xgb
from xgboost import XGBClassifier

In [2]:
def load_training_dataframe(parquet_root: Union[str, Path]) -> pd.DataFrame:
    """Load and prepare parquet files for model training.

    Parameters
    ----------
    parquet_root:
        Directory containing parquet files. All parquet files found recursively
        under this directory are concatenated into a single DataFrame.

    Returns
    -------
    pd.DataFrame
        Data ready for XGBoost. Rows with missing values are dropped and
        columns are converted to integer or boolean types where appropriate.
    """

    parquet_root = Path(parquet_root)
    files = sorted(parquet_root.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found under {parquet_root}")

    # Read all parquet files and concatenate
    frames = [pd.read_parquet(f, engine="pyarrow") for f in files]
    df = pd.concat(frames, ignore_index=True)

    # Drop any rows containing NA values
    df = df.dropna().reset_index(drop=True)

    # Convert object columns to numeric or categorical codes
    obj_cols = df.select_dtypes(include="object").columns 
    obj_cols = obj_cols.drop("match_id")
    for col in obj_cols:
        lower = df[col].str.lower()
        if set(lower.unique()) <= {"true", "false"}:
            df[col] = lower == "true"
        else:
            df[col] = df[col].astype("category").cat.codes

    # Convert numeric columns to int or bool where possible
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        series = df[col]
        if pd.api.types.is_float_dtype(series) and np.allclose(series, series.astype(int)):
            series = series.astype(int)
        if set(series.unique()) <= {0, 1}:
            series = series.astype(bool)
        else:
            series = pd.to_numeric(series, downcast="integer")
        df[col] = series

    return df

In [3]:
dfs = [pd.read_parquet(f) for f in glob.glob("data/matches/*.parquet")]
match_df = dfs[np.random.randint(len(dfs))]

In [ ]:
data = load_training_dataframe("data/matches")

In [ ]:
#rand_id = (data['match_id'].unique())[np.random.randint(len(data['match_id'].unique()))]
rand_id = '2023-usopen-1131'
match_df = data[data['match_id'] == rand_id]
display(match_df)

,match_id,SetNo,GameNo,PointNumber,point_idx,perspective,server_is_persp,pts_in_game_for,pts_in_game_against,games_in_set_for,...,best_of,sets_needed_to_win,is_tiebreak,is_game_point_for,is_game_point_against,is_break_point,ttl_diff,aces_diff,df_diff,y_match
3051486,2023-usopen-1131,1,1,500,1,False,False,0,0,0,...,5,3,False,False,False,False,0,0,0,False
3051487,2023-usopen-1131,1,1,501,2,False,False,0,0,0,...,5,3,False,False,False,False,0,0,0,False
3051488,2023-usopen-1131,1,1,502,3,False,False,0,0,0,...,5,3,False,False,False,False,0,0,0,False
3051489,2023-usopen-1131,1,1,703,4,False,False,0,1,0,...,5,3,False,False,False,False,-1,0,0,False
3051490,2023-usopen-1131,1,1,844,5,False,False,0,2,0,...,5,3,False,False,False,False,-2,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3051993,2023-usopen-1131,4,13,779,252,True,False,3,3,6,...,5,3,True,False,False,False,5,18,1,True
3051994,2023-usopen-1131,4,13,780,253,True,True,4,3,6,...,5,3,True,False,False,False,6,18,1,True
3051995,2023-usopen-1131,4,13,781,254,True,True,5,3,6,...,5,3,True,False,False,False,7,18,1,True
3051996,2023-usopen-1131,4,13,782,255,True,False,6,3,6,...,5,3,True,False,False,False,8,18,1,True


In [ ]:
def predict_win_probability(match_df, rand_id, model, feature_cols):
    matches = data['match_id'].unique()
    feature_cols = [c for c in data.columns if c not in ('match_id', 'y_match')]
    train_ids = set(np.random.choice(matches, size=int(0.8*len(matches)), replace=False))
    train_data = data[data['match_id'].isin(train_ids)]
    test_data  = data[~data['match_id'].isin(train_ids)]
    X_train, y_train = train_data[feature_cols], train_data['y_match']
    X_test,  y_test  = test_data[feature_cols],  test_data['y_match']
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    print("Log Loss:", log_loss(y_test, y_pred_prob))
    print("ROC AUC:", roc_auc_score(y_test, y_pred_prob))

    win_prob = []
    for _,point in match_df.iterrows():
        X_point = point[feature_cols].values.reshape(1, -1)
        prob = model.predict_proba(X_point)[0, 1]  # probability Player 1 wins match
        win_prob.append(prob)
    plt.plot(win_prob[:len(win_prob)//2], label='Win Probability (Player 1)')
    plt.xlabel('Point Index')
    plt.ylabel('Probability Player Wins Match')

    plt.plot(win_prob[len(win_prob)//2:], label='Win Probability (Player 2)')
    plt.title(f'Live Win Probability Prediction (Match ID: {rand_id})')
    plt.ylim(0, 1)
    plt.legend()
    plt.show()



In [ ]:
model_df = pd.read_parquet("model_data.parquet", engine="pyarrow")
for i in range(len(model_df)):
    model = XGBClassifier()
    model.load_model(model_df['model_path'][i])
    feature_cols = model_df['input_columns'][i].split('$')
    predict_win_probability(match_df, rand_id, model, feature_cols)

In [ ]:
print(win_prob[0])

0.49773526
